# Лабораторная работа 12. Квантизация и speculative decoding (студенческая версия)

**Курс:** Методы обработки естественного языка · **Блок 4:** инференс и эффективность

**Цель.** Реализовать два основных способа ускорить инференс без переобучения модели и проверить их теоретические свойства.

**Что нужно сдать:**
1. Absmax- и zero-point-квантизацию int8/int4 с разной гранулярностью и таблицу perplexity.
2. Построение NF4 и сравнение ошибки квантизации с int4 на реальных весах.
3. Анализ выбросов в активациях и их влияния на квантизацию активаций.
4. Упрощённый GPTQ для одного линейного слоя и сравнение с round-to-nearest.
5. Speculative sampling: эмпирическую проверку совпадения распределений (критерий хи-квадрат) и измерение ускорения.

**Теория.**
- **Absmax (симметричная):** $s = \max|w| / (2^{b-1}-1)$, $\hat w = s \cdot \mathrm{round}(w/s)$.
- **Zero-point (асимметричная):** $s = (\max w - \min w)/(2^b - 1)$, $z = \mathrm{round}(-\min w / s)$, $\hat w = s(\mathrm{clamp}(\mathrm{round}(w/s) + z) - z)$.
- **GPTQ** минимизирует $\|XW^\top - X\hat W^\top\|^2$ по столбцам: после квантизации столбца $j$ ошибка $\delta_j = (w_j - \hat w_j)/[H^{-1}]_{jj}$ компенсируется обновлением ещё не квантованных столбцов $w_{j+1:} \mathrel{-}= \delta_j [H^{-1}]_{j,j+1:}$, где $H = 2X^\top X$ (Frantar et al., 2023; OBS).
- **Speculative sampling:** draft-модель $q$ предлагает токен $x$, целевая модель $p$ принимает его с вероятностью $\min(1, p(x)/q(x))$; при отказе токен берётся из $\mathrm{norm}(\max(0, p - q))$. Итоговое распределение в точности равно $p$.


In [1]:
# ==== Общая настройка (одинакова во всех работах) ====
import os, math, time, json, random, pathlib, urllib.request
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

FAST_DEV_RUN = True          # True — сокращённый прогон (CPU, минуты); False — полный эксперимент
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ARTIFACTS = pathlib.Path("artifacts"); ARTIFACTS.mkdir(exist_ok=True)
DATA = pathlib.Path("data"); DATA.mkdir(exist_ok=True)

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
set_seed()

def download(url, name):
    path = DATA / name
    if not path.exists():
        urllib.request.urlretrieve(url, path)
    return path

def load_shakespeare():
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    return download(url, "shakespeare.txt").read_text(encoding="utf-8")

def load_russian():
    """Русский корпус: «Война и мир» (Л. Н. Толстой), т. 1, из открытого репозитория.
    При недоступности сети используется встроенный фрагмент."""
    urls = ["https://raw.githubusercontent.com/mmcdermott/nlp-war-and-peace/master/data/war_and_peace_ru.txt",
            "https://raw.githubusercontent.com/nevmenandr/word2vec-russian-novels/master/books_before/WarAndPeace.txt"]
    for i, u in enumerate(urls):
        try:
            t = download(u, f"ru_{i}.txt").read_text(encoding="utf-8", errors="ignore")
            if len(t) > 100_000: return t
        except Exception:
            pass
    base = ("Все счастливые семьи похожи друг на друга, каждая несчастливая семья несчастлива по-своему. "
            "Всё смешалось в доме Облонских. Жена узнала, что муж был в связи с бывшею в их доме француженкою-гувернанткой, "
            "и объявила мужу, что не может жить с ним в одном доме. ")
    return base * 400

print(f"torch {torch.__version__}, device={DEVICE}, FAST_DEV_RUN={FAST_DEV_RUN}")

torch 2.12.0.dev20260408+cu128, device=cuda, FAST_DEV_RUN=True


In [2]:
# ==== Загрузка предобученных моделей Hugging Face ====
from transformers import AutoTokenizer, AutoModelForCausalLM

def load_lm(name, dtype=None):
    tok = AutoTokenizer.from_pretrained(name)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    dtype = dtype or (torch.float16 if DEVICE == "cuda" else torch.float32)
    try:
        model = AutoModelForCausalLM.from_pretrained(name, dtype=dtype)
    except TypeError:
        model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=dtype)
    return model.to(DEVICE).eval(), tok

@torch.no_grad()
def perplexity(model, tok, text, max_len=512, stride=256, max_windows=None):
    ids = tok(text, return_tensors="pt").input_ids.to(DEVICE)
    nll, cnt, prev = 0.0, 0, 0
    for i, start in enumerate(range(0, ids.size(1) - 1, stride)):
        if max_windows and i >= max_windows: break
        end = min(start + max_len, ids.size(1)); x = ids[:, start:end]
        tgt = x.clone(); tgt[:, : max(0, prev - start)] = -100
        out = model(x, labels=tgt)
        n = (tgt[:, 1:] != -100).sum().item(); nll += out.loss.float().item() * n; cnt += n; prev = end
        if end == ids.size(1): break
    return math.exp(nll / cnt)

e:\temp\llm\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
TARGET, DRAFT = ("HuggingFaceTB/SmolLM2-360M", "HuggingFaceTB/SmolLM2-135M") if FAST_DEV_RUN else ("Qwen/Qwen2.5-1.5B", "Qwen/Qwen2.5-0.5B")
model, tok = load_lm(DRAFT, dtype=torch.float32)          # квантизация исследуется на меньшей модели
eval_text = load_shakespeare()[-(20_000 if FAST_DEV_RUN else 200_000):]
MAXW = 8 if FAST_DEV_RUN else None
base_ppl = perplexity(model, tok, eval_text, max_windows=MAXW)
print(f"{DRAFT}: perplexity FP32 = {base_ppl:.2f}")

Loading weights: 100%|██████████| 272/272 [00:00<00:00, 992.27it/s]


HuggingFaceTB/SmolLM2-135M: perplexity FP32 = 27.11


## 1. Absmax и zero-point квантизация весов

Реализуйте fake-quantization (квантизация + деквантизация) с гранулярностью:
- `per_tensor` — один масштаб на матрицу;
- `per_channel` — масштаб на выходной канал (строку);
- `group` — масштаб на группу из `group_size` подряд идущих входных весов строки.

Квантуйте все линейные слои трансформера (кроме эмбеддингов и выходной головы) и измерьте perplexity.

In [ ]:
def quantize(w, bits=8, mode="absmax", granularity="per_channel", group_size=128):
    shape = w.shape
    if granularity == "per_tensor":
        x = w.reshape(1, -1)
    elif granularity == "per_channel":
        x = w.reshape(shape[0], -1)
    else:
        x = w.reshape(-1, group_size)

    if mode == "absmax":
        qmax = (2 ** (bits - 1)) - 1
        qmin = -(2 ** (bits - 1))
        
        scale = x.abs().amax(dim=-1, keepdim=True) / qmax
        scale = torch.clamp(scale, min=1e-8)
        
        q = torch.clamp(torch.round(x / scale), qmin, qmax)
        xq = q * scale

    elif mode == "zeropoint":
        qmin = 0
        qmax = (2 ** bits) - 1
        
        min_val = x.amin(dim=-1, keepdim=True)
        max_val = x.amax(dim=-1, keepdim=True)
        
        scale = torch.clamp((max_val - min_val) / (qmax - qmin), min=1e-8)
        zero_point = torch.clamp(torch.round(qmin - min_val / scale), qmin, qmax)
        
        q = torch.clamp(torch.round(x / scale + zero_point), qmin, qmax)
        xq = (q - zero_point) * scale

    return xq.reshape(shape)

import copy
def quantize_model(model, fn):
    m = copy.deepcopy(model)
    with torch.no_grad():
        for name, mod in m.named_modules():
            if isinstance(mod, nn.Linear) and "lm_head" not in name:
                mod.weight.copy_(fn(mod.weight.data))
    return m

w = torch.randn(256, 512)
for b in (8, 4):
    for g in ("per_tensor", "per_channel", "group"):
        print(f"int{b} {g:12s} rel. MSE = {((quantize(w, b, granularity=g) - w).pow(2).mean() / w.pow(2).mean()).item():.2e}")

int8 per_tensor   rel. MSE = 1.04e-04
int8 per_channel  rel. MSE = 5.52e-05
int8 group        rel. MSE = 4.19e-05
int4 per_tensor   rel. MSE = 3.43e-02
int4 per_channel  rel. MSE = 1.82e-02
int4 group        rel. MSE = 1.39e-02


In [12]:
import pandas as pd
rows = [{"схема": "FP32", "perplexity": base_ppl}]
configs = [(8, "absmax", "per_tensor"), (8, "absmax", "per_channel"), (4, "absmax", "per_tensor"), (4, "absmax", "per_channel"),
           (4, "absmax", "group"), (4, "zeropoint", "per_channel"), (4, "zeropoint", "group"), (3, "absmax", "group")]

for bits, mode, gran in configs:
    qm = quantize_model(
        model,
        lambda w, bits=bits, mode=mode, gran=gran: quantize(
            w, bits=bits, mode=mode, granularity=gran
        ),
    )
    ppl = perplexity(qm, tok, eval_text, max_windows=MAXW)
    rows.append({"схема": f"int{bits} {mode} {gran}", "perplexity": ppl})
    del qm

display(pd.DataFrame(rows).round(3))

,схема,perplexity
0,FP32,27.110
1,int8 absmax per_tensor,28.745
2,int8 absmax per_channel,27.152
3,int4 absmax per_tensor,3733550.451
4,int4 absmax per_channel,79.904
5,int4 absmax group,46.832
6,int4 zeropoint per_channel,55.148
7,int4 zeropoint group,40.267
8,int3 absmax group,921.594


## 2. NF4

NF4 (Dettmers et al., 2023) — 16 уровней, равных квантилям стандартного нормального распределения, нормированных на \([-1, 1]\), с точным нулём: 8 положительных уровней и 7 отрицательных плюс 0. Квантизация блочная: масштаб — absmax блока из 64 весов, каждое нормированное значение заменяется ближайшим уровнем.

Сравните относительную MSE для NF4 и int4 absmax (блок 64) на весах реальной модели и на синтетических данных: гауссовых и равномерных.

In [ ]:
from scipy.stats import norm

def nf4_levels(offset=0.9677):
    q_neg = np.linspace(1 - offset, 0.5, 8)[:-1]
    q_pos = np.linspace(0.5, offset, 9)[1:]
    
    v_neg = norm.ppf(q_neg)
    v_pos = norm.ppf(q_pos)
    
    v_neg = v_neg / np.abs(v_neg).max()
    v_pos = v_pos / v_pos.max()
    
    levels = np.concatenate([v_neg, [0.0], v_pos])
    return torch.tensor(levels, dtype=torch.float32)

NF4 = nf4_levels(); print("уровни NF4:", NF4.numpy().round(4)); assert len(NF4) == 16 and (NF4 == 0).any()

def quantize_nf4(w, block=64):
    shape = w.shape
    device, dtype = w.device, w.dtype
    x = w.reshape(-1, block).to(torch.float32)
    
    scale = x.abs().amax(dim=-1, keepdim=True)
    scale = torch.clamp(scale, min=1e-8)
    x_norm = x / scale
    
    levels = NF4.to(device=device, dtype=torch.float32)
    idx = (x_norm.unsqueeze(-1) - levels).abs().argmin(dim=-1)
    xq = levels[idx] * scale
    
    return xq.reshape(shape).to(dtype)

def rel_mse(a, b): return ((a - b).pow(2).mean() / b.pow(2).mean()).item()
W_real = torch.cat([m.weight.data.flatten()[: 2 ** 16] for n, m in model.named_modules() if isinstance(m, nn.Linear) and "lm_head" not in n][:20])
for name, wt in [("реальные веса", W_real), ("N(0,1)", torch.randn(2 ** 18)), ("U(-1,1)", torch.rand(2 ** 18) * 2 - 1)]:
    print(f"{name:14s} int4 absmax: {rel_mse(quantize(wt, 4, granularity='group', group_size=64), wt):.4f}   NF4: {rel_mse(quantize_nf4(wt), wt):.4f}")
qm = quantize_model(model, quantize_nf4); print("perplexity NF4:", round(perplexity(qm, tok, eval_text, max_windows=MAXW), 2)); del qm

уровни NF4: [-1.     -0.6962 -0.5251 -0.3949 -0.2845 -0.1848 -0.0911  0.      0.0796
  0.1609  0.2461  0.3379  0.4407  0.5626  0.723   1.    ]
реальные веса  int4 absmax: 0.0134   NF4: 0.0090
N(0,1)         int4 absmax: 0.0115   NF4: 0.0084
U(-1,1)        int4 absmax: 0.0049   NF4: 0.0083
perplexity NF4: 35.16


## 3. Выбросы в активациях

Соберите хуком входы линейных слоёв `down_proj` (или аналогичных) и посчитайте максимум \(|x|\) по каждому входному каналу. Найдите каналы-выбросы (максимум в 20 и более раз выше медианы по каналам). Сравните ошибку выхода слоя при:
1. int8 per-tensor квантизации активаций;
2. int8 per-token квантизации;
3. разложении LLM.int8(): каналы-выбросы остаются в FP, остальные — int8 per-token.

In [ ]:
acts = {}
def save_input(name):
    def hook(mod, inp, out): acts.setdefault(name, []).append(inp[0].detach().reshape(-1, inp[0].size(-1)))
    return hook
hooks = [m.register_forward_hook(save_input(n)) for n, m in model.named_modules() if n.endswith("down_proj")]
calib_ids = tok(eval_text, return_tensors="pt").input_ids[0]
calib_ids = calib_ids[: len(calib_ids) // 512 * 512].view(-1, 512)[:8]      # до 8 окон по 512 токенов (около 4 тыс. токенов)
with torch.no_grad():
    for w in calib_ids: model(w[None].to(DEVICE))
for h in hooks: h.remove()
acts = {k: torch.cat(v) for k, v in acts.items()}

# (статистика выбросов по слоям и сравнение трёх схем на слое с наибольшими выбросами)
stats = []
for name, X in acts.items():
    X = X.float()
    cmax = X.abs().amax(dim=0)
    med = cmax.median().clamp(min=1e-12)
    out_ch = cmax >= 20 * med
    stats.append({
        "Слой": name,
        "Токенов": X.size(0),
        "Каналов (d_in)": X.size(1),
        "Медиана max|X|": med.item(),
        "Максимум |X|": cmax.max().item(),
        "Кратность (max / median)": (cmax.max() / med).item(),
        "Каналов-выбросов": int(out_ch.sum()),
        "Доля выбросов, %": (out_ch.float().mean() * 100).item(),
    })

stats_df = pd.DataFrame(stats).sort_values(
    ["Каналов-выбросов", "Кратность (max / median)"], ascending=False
)
display(stats_df.round(4))

worst = stats_df.iloc[0]["Слой"]
print(f"Слой с наибольшими выбросами: {worst}\n")

X = acts[worst].float()
W = dict(model.named_modules())[worst].weight.data.float()
ref = X @ W.T

X_pt = quantize(X, 8, mode="absmax", granularity="per_tensor")
X_tok = quantize(X, 8, mode="absmax", granularity="per_channel")

cmax = X.abs().amax(dim=0)
med = cmax.median().clamp(min=1e-12)
out_ch = cmax >= 20 * med
non_out = ~out_ch

X_mix = X.clone()
X_mix[:, non_out] = quantize(
    X[:, non_out], 8, mode="absmax", granularity="per_channel"
)

for scheme_name, Xq in [
    ("1. int8 per-tensor (на весь тензор)", X_pt),
    ("2. int8 per-token (построчно)", X_tok),
    ("3. LLM.int8() (смешанная точность)", X_mix),
]:
    print(f"{scheme_name:38s} | rel. MSE = {rel_mse(Xq @ W.T, ref):.4e}")



,Слой,Токенов,Каналов (d_in),Медиана max|X|,Максимум |X|,Кратность (max / median),Каналов-выбросов,"Доля выбросов, %"
11,model.layers.11.mlp.down_proj,4096,1536,1.4523,2478.8428,1706.8334,2,0.1302
28,model.layers.28.mlp.down_proj,4096,1536,7.5033,2649.4517,353.1052,2,0.1302
2,model.layers.2.mlp.down_proj,4096,1536,1.0758,83.2950,77.4256,2,0.1302
29,model.layers.29.mlp.down_proj,4096,1536,9.0124,469.4836,52.0931,1,0.0651
9,model.layers.9.mlp.down_proj,4096,1536,1.4782,72.7048,49.1835,1,0.0651
10,model.layers.10.mlp.down_proj,4096,1536,1.5122,33.2795,22.0074,1,0.0651
17,model.layers.17.mlp.down_proj,4096,1536,1.4275,18.1493,12.7144,0,0.0000
23,model.layers.23.mlp.down_proj,4096,1536,4.1790,45.9664,10.9994,0,0.0000
12,model.layers.12.mlp.down_proj,4096,1536,1.2400,13.5971,10.9656,0,0.0000
21,model.layers.21.mlp.down_proj,4096,1536,2.6687,27.9055,10.4566,0,0.0000


Слой с наибольшими выбросами: model.layers.11.mlp.down_proj

1. int8 per-tensor (на весь тензор)    | rel. MSE = 1.2314e-03
2. int8 per-token (построчно)          | rel. MSE = 4.6377e-05
3. LLM.int8() (смешанная точность)     | rel. MSE = 9.0039e-07


## 4. Упрощённый GPTQ для одного слоя

Возьмите слой `down_proj` из п. 3 и калибровочные активации $X$: первые 3000 токенов — калибровка, остальные — отложенная выборка. Число калибровочных токенов должно заметно превышать $d_{in}$ (1536 у `down_proj` SmolLM2-135M), иначе $H$ вырождена и GPTQ переобучается на калибровку. Реализуйте GPTQ без ленивых блочных обновлений:
1. $H = 2X^\top X + \lambda I$, $\lambda = 0.01 \cdot \mathrm{mean}(\mathrm{diag}\,H)$;
2. $U = \mathrm{chol}(H^{-1})^\top$ (верхнетреугольная), так что $[H^{-1}]_{jj}$ и строки обновления берутся из $U$;
3. для $j = 1..d_{in}$: квантовать столбец $w_{:,j}$ (int4, per-channel масштаб, заранее посчитанный по исходной матрице), $\delta = (w_{:,j} - \hat w_{:,j})/U_{jj}$, $W_{:,j+1:} \mathrel{-}= \delta\,U_{j,j+1:}$.

Сравните $\|XW^\top - X\hat W^\top\|^2 / \|XW^\top\|^2$ для RTN и GPTQ на калибровочной и на отложенной выборке активаций.


In [ ]:
def gptq_layer(W, X, bits=4, damp=0.01):
    W = W.clone().float()
    d_in = W.size(1)
    qmax = 2 ** (bits - 1) - 1
    scale = W.abs().amax(1, keepdim=True).clamp(min=1e-8) / qmax
    q = lambda col: (col / scale[:, 0]).round().clamp(-qmax, qmax) * scale[:, 0]

    H = 2.0 * (X.T @ X)
    lam = damp * torch.diag(H).mean()
    idx = torch.arange(d_in, device=W.device)
    H[idx, idx] += lam

    H_inv = torch.linalg.inv(H)
    U = torch.linalg.cholesky(H_inv).T

    for j in range(d_in):
        w_j = W[:, j].clone()
        w_hat_j = q(w_j)
        W[:, j] = w_hat_j

        delta = (w_j - w_hat_j) / U[j, j]
        if j + 1 < d_in:
            W[:, j + 1:] -= delta.unsqueeze(1) * U[j, j + 1:].unsqueeze(0)

    return W

lin = dict(model.named_modules())[worst]; Wl = lin.weight.data.float()
X_all = acts[worst].float(); X_cal, X_hold = X_all[:3000], X_all[3000:]
t0 = time.time(); W_gptq = gptq_layer(Wl, X_cal); print(f"GPTQ: {time.time() - t0:.1f} c")
W_rtn = quantize(Wl, 4, granularity="per_channel")
for name, Xe in [("калибровка", X_cal), ("отложенная", X_hold)]:
    ref = Xe @ Wl.T
    print(f"{name:11s} RTN: {rel_mse(Xe @ W_rtn.T, ref):.4e}   GPTQ: {rel_mse(Xe @ W_gptq.T, ref):.4e}")

GPTQ: 0.3 c
калибровка  RTN: 3.1419e-03   GPTQ: 8.8403e-04
отложенная  RTN: 3.1336e-03   GPTQ: 1.1086e-03


## 5. Speculative sampling

### 5а. Проверка на игрушечных распределениях

Реализуйте один шаг speculative sampling для заданных распределений $p$ (цель) и $q$ (черновик) над 10 символами. Повторите 100 000 раз и проверьте критерием хи-квадрат, что распределение выхода совпадает с $p$. Посчитайте долю принятых токенов и сравните с теоретической $\sum_x \min(p(x), q(x))$.


In [ ]:
from scipy.stats import chisquare

def chi2_pvalue(counts, probs):
    """χ²-тест согласия; ожидаемые частоты нормируются точно к сумме наблюдений (иначе scipy ругается на округление)."""
    counts = np.asarray(counts, dtype=float); e = np.asarray(probs, dtype=float); e = e / e.sum() * counts.sum()
    return chisquare(counts, e).pvalue

def spec_step(p, q, gen):
    """Один шаг: возвращает (токен, принят ли черновик)."""
    x = torch.multinomial(q, 1, generator=gen).item()

    qx = q[x].item()
    px = p[x].item()
    accept_prob = min(1.0, px / max(qx, 1e-12))

    u = torch.rand((), generator=gen).item()
    if u <= accept_prob:
        return x, True

    resid = torch.clamp(p - q, min=0)
    if resid.sum() <= 0:
        resid = p.clone()
    y = torch.multinomial(resid / resid.sum(), 1, generator=gen).item()
    return y, False

gen = torch.Generator().manual_seed(0)
p = torch.distributions.Dirichlet(torch.ones(10)).sample(); q = torch.distributions.Dirichlet(torch.ones(10)).sample()
N = 100_000 if not FAST_DEV_RUN else 30_000
res = [spec_step(p, q, gen) for _ in range(N)]
counts = np.bincount([r[0] for r in res], minlength=10)
print("χ²-тест против p: p-value =", round(chi2_pvalue(counts, p.numpy()), 3))
print("χ²-тест против q: p-value =", chi2_pvalue(counts, q.numpy()))
print(f"доля принятых: {np.mean([r[1] for r in res]):.4f}, теория: {torch.minimum(p, q).sum().item():.4f}")

χ²-тест против p: p-value = 0.052
χ²-тест против q: p-value = 0.0
доля принятых: 0.5114, теория: 0.5124


### 5б. Speculative decoding для LLM

Draft — меньшая модель того же семейства (общий токенизатор). Алгоритм для длины черновика $\gamma$:
1. draft авторегрессионно генерирует $\gamma$ токенов, сохраняя $q_i$;
2. target одним прямым проходом вычисляет $p_i$ для всех $\gamma + 1$ позиций;
3. токены принимаются слева направо по правилу из 5а; при первом отказе генерируется исправленный токен и шаг заканчивается; если все приняты — добавляется токен из $p_{\gamma+1}$.

Для простоты KV-cache не используется: каждый вызов модели — полный проход по префиксу. Измерьте число вызовов target на сгенерированный токен, долю принятых токенов и wall-clock время в сравнении с обычной генерацией target-моделью при $\gamma \in \{2, 4, 6\}$.


In [ ]:
target, _ = load_lm(TARGET); draft = model.to(torch.float16 if DEVICE == "cuda" else torch.float32)

@torch.no_grad()
def probs_of(m, ids): return m(ids).logits[0].float().softmax(-1)

@torch.no_grad()
def speculative_generate(ids, n_new, gamma=4, gen=None):
    stats = {"target_calls": 0, "draft_calls": 0, "accepted": 0, "proposed": 0}
    start = ids.size(1)

    while ids.size(1) - start < n_new:
        draft_tokens = []
        q_probs = []
        draft_ids = ids

        for _ in range(gamma):
            q = probs_of(draft, draft_ids)[-1]
            q_probs.append(q)

            x = torch.multinomial(q, 1, generator=gen)[None]
            draft_tokens.append(x)
            draft_ids = torch.cat([draft_ids, x], dim=1)
            stats["draft_calls"] += 1

        draft_tokens = torch.cat(draft_tokens, dim=1)
        full_ids = torch.cat([ids, draft_tokens], dim=1)

        L = ids.size(1)
        p_all = probs_of(target, full_ids)[L - 1: L + gamma]
        stats["target_calls"] += 1

        all_accepted = True

        for i in range(gamma):
            stats["proposed"] += 1

            x = draft_tokens[0, i].item()
            pi = p_all[i]
            qi = q_probs[i]

            ratio = (pi[x] / qi[x].clamp(min=1e-12)).item()
            u = torch.rand((), generator=gen, device=ids.device).item()

            if u <= min(1.0, ratio):
                ids = torch.cat([ids, draft_tokens[:, i:i + 1]], dim=1)
                stats["accepted"] += 1
            else:
                resid = torch.clamp(pi - qi, min=0)
                if resid.sum() <= 0:
                    resid = pi.clone()

                y = torch.multinomial(resid / resid.sum(), 1, generator=gen)[None]
                ids = torch.cat([ids, y], dim=1)
                all_accepted = False
                break

        if all_accepted:
            y = torch.multinomial(p_all[gamma], 1, generator=gen)[None]
            ids = torch.cat([ids, y], dim=1)

    return ids[:, :start + n_new], stats

@torch.no_grad()
def plain_generate(m, ids, n_new, gen=None):
    for _ in range(n_new): ids = torch.cat([ids, torch.multinomial(probs_of(m, ids)[-1], 1, generator=gen)[None]], 1)
    return ids

prompt = tok("The history of the city begins", return_tensors="pt").input_ids.to(DEVICE)
N_NEW = 40 if FAST_DEV_RUN else 100
g = torch.Generator(device=DEVICE).manual_seed(0) if DEVICE == "cuda" else torch.Generator().manual_seed(0)
t0 = time.time(); plain_generate(target, prompt, N_NEW, g); t_plain = time.time() - t0
rows = [{"метод": "target, обычная генерация", "время, c": t_plain, "вызовов target на токен": 1.0, "доля принятых": float("nan")}]
for gamma in (2, 4, 6):
    t0 = time.time(); out, st = speculative_generate(prompt, N_NEW, gamma, g); dt = time.time() - t0
    rows.append({"метод": f"speculative, γ={gamma}", "время, c": dt, "вызовов target на токен": st["target_calls"] / N_NEW,
                 "доля принятых": st["accepted"] / st["proposed"]})
display(pd.DataFrame(rows).round(3)); print(tok.decode(out[0]))

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 637.86it/s]


,метод,"время, c",вызовов target на токен,доля принятых
0,"target, обычная генерация",1.857,1.000,NaN
1,"speculative, γ=2",2.130,0.425,0.781
2,"speculative, γ=4",2.848,0.350,0.722
3,"speculative, γ=6",3.882,0.350,0.718


The history of the city begins only slightly later. It was founded in around 300 BC by Pagans who took the land of two of the city’s neighbours.

Among the ruins, sponge divers and reeds


### 5в. Проверка распределения на реальных моделях

Для фиксированного промпта сгенерируйте первый токен 150 раз (в полном режиме 2000) обычным сэмплированием из target и speculative-методом (\(\gamma = 1\)). Сравните частоты с \(p\) критерием хи-квадрат, объединив токены вне top-10 в одну категорию.

In [28]:
prompt_5v = prompt
M = 2000

p_exact = probs_of(target, prompt_5v)[-1].cpu()

top10_ids = torch.topk(p_exact, 10).indices.tolist()
top10_map = {tok_id: i for i, tok_id in enumerate(top10_ids)}

p_top10 = p_exact[top10_ids].numpy()
p_bins = np.append(p_top10, max(0.0, 1.0 - p_top10.sum()))

def to_11_bins(sampled_ids):
    counts = np.zeros(11, dtype=float)
    for t in sampled_ids:
        counts[top10_map.get(t, 10)] += 1
    return counts

gen_5v = torch.Generator(device=DEVICE).manual_seed(42) if DEVICE == "cuda" else torch.Generator().manual_seed(42)

plain_samples = [
    plain_generate(target, prompt_5v, 1, gen_5v)[0, -1].item()
    for _ in range(M)
]

spec_samples = [
    speculative_generate(prompt_5v, 1, gamma=1, gen=gen_5v)[0][0, -1].item()
    for _ in range(M)
]

counts_plain = to_11_bins(plain_samples)
counts_spec = to_11_bins(spec_samples)

print(f"Число испытаний M = {M}")
print(f"χ²-тест (обычный target vs p):      p-value = {chi2_pvalue(counts_plain, p_bins):.4f}")
print(f"χ²-тест (speculative γ=1 vs p):     p-value = {chi2_pvalue(counts_spec, p_bins):.4f}")


Число испытаний M = 2000
χ²-тест (обычный target vs p):      p-value = 0.6134
χ²-тест (speculative γ=1 vs p):     p-value = 0.6594


## Вопросы

1. Почему per-channel квантизация весов почти не теряет качества по сравнению с per-tensor, а для активаций нужна per-token и обработка выбросов?
2. Для какого распределения весов NF4 оптимальна и в каком смысле? Почему на равномерном распределении она уступает int4?
3. Выведите правило обновления GPTQ из задачи OBS: минимизировать прирост ошибки при фиксации одного веса.
4. Докажите, что speculative sampling сохраняет распределение \(p\). Выведите ожидаемое число токенов на один вызов target при вероятности принятия \(\alpha\) и длине черновика \(\gamma\).
5. Почему без KV-cache ускорение по wall-clock меньше, чем по числу вызовов target? Когда speculative decoding не даёт ускорения?

Выводы: сделали квантизацию весов модели, сжали числа до int8/int4 разными способами и проверили, как это влияет на perplexity. Сравнили NF4 с обычной int4 и увидели, где NF4 лучше, а где хуже. Посмотрели, что в активациях есть выбросы, и проверили, как с ними справляются разные схемы квантизации. Сделали GPTQ для одного слоя и показали, что он ошибается меньше, чем простое округление. Проверили speculative sampling, доказали, что распределение не меняется, и замерили скорость на реальных моделях.